# Microbiota-only logistic regression

Figure 5b–d and Supplementary Figure 4. The ERG workbook is required to reconstruct the original joined, complete-case cohort. ERG columns determine cohort eligibility but are excluded from the fitted classifiers. The source reports balanced accuracy at the pooled out-of-fold Youden threshold.

In [ ]:
from pathlib import Path

# Jupyter normally starts the kernel in the notebook directory.
here = Path.cwd()
repo_root = next(parent for parent in (here, *here.parents)
                 if (parent / "requirements.txt").is_file()
                 and (parent / "mouse").is_dir() and (parent / "human").is_dir())
data_dir = repo_root / "data"
output_dir = repo_root / "outputs" / 'human/06_saliva_logistic_regression'
output_dir.mkdir(parents=True, exist_ok=True)
notebook_output = output_dir


## Load and join the cohort workbooks

In [ ]:
import pandas as pd

# Define the file paths
erg_path = data_dir / 'Human_RawData_for_PCA_ERG.xlsx'
saliva_path = data_dir / 'Human_RawData_for_PCA_SALIVA.xlsx'

# Load the spreadsheets as dataframes
df_erg = pd.read_excel(erg_path)
df_saliva = pd.read_excel(saliva_path)

# Ensure SampleID is a string before slicing
df_saliva["SampleID"] = df_saliva["SampleID"].astype(str).str[:6]

# Then merge the two DataFrames on SampleID
merged_df = pd.merge(df_erg, df_saliva, on="SampleID", how="inner")


In [ ]:
import pandas as pd
import numpy as np

# -------------------------
# 1. Define feature groups
# -------------------------

erg_features = [col for col in df_erg.columns if col != "SampleID"]

saliva_metadata_cols = [
    'SampleID', 'Age', 'Gender', 'Treatment', 'Disease_Duration',
    'Symptom_Duration', 'H_Y_Score', 'Collection'
]
microbiota_features = [col for col in df_saliva.columns if col not in saliva_metadata_cols]

print(f"ERG features: {len(erg_features)}")
print(f"Microbiota features: {len(microbiota_features)}")

# -------------------------
# 2. Keep only control vs PD
# -------------------------

analysis_df = merged_df.copy()
analysis_df["Treatment_lower"] = analysis_df["Treatment"].astype(str).str.lower().str.strip()

analysis_df = analysis_df[
    analysis_df["Treatment_lower"].isin(["control", "parkinson", "pd", "p"])
].copy()

analysis_df["PD_status"] = analysis_df["Treatment_lower"].map({
    "control": 0,
    "parkinson": 1,
    "pd": 1,
    "p": 1
})

print(analysis_df["PD_status"].value_counts())

# -------------------------
# 3. Force feature columns numeric where possible
# -------------------------

for col in erg_features + microbiota_features:
    if col in analysis_df.columns:
        analysis_df[col] = pd.to_numeric(analysis_df[col], errors="coerce")

# Optional: inspect missingness
missing_summary = analysis_df[erg_features + microbiota_features].isna().mean().sort_values(ascending=False)
print("\nTop missingness:")
print(missing_summary.head(10))


## Rank candidate features for the original cohort

In [ ]:
from scipy.stats import ttest_ind
from sklearn.metrics import roc_auc_score
from statsmodels.stats.multitest import multipletests

def screen_features(df, features, outcome_col="PD_status"):
    results = []

    for feature in features:
        sub = df[[feature, outcome_col]].dropna()
        if sub.empty:
            continue

        x0 = sub.loc[sub[outcome_col] == 0, feature]
        x1 = sub.loc[sub[outcome_col] == 1, feature]

        if len(x0) < 2 or len(x1) < 2:
            continue

        # Welch t-test
        stat, pval = ttest_ind(x0, x1, equal_var=False)

        # Means
        mean_ctrl = x0.mean()
        mean_pd = x1.mean()
        mean_diff = mean_pd - mean_ctrl

        # ROC AUC
        try:
            auc = roc_auc_score(sub[outcome_col], sub[feature])
            # Also compute direction-independent AUC
            auc_abs = max(auc, 1 - auc)
        except:
            auc = np.nan
            auc_abs = np.nan

        results.append({
            "Feature": feature,
            "N": len(sub),
            "N_Control": len(x0),
            "N_PD": len(x1),
            "Mean_Control": mean_ctrl,
            "Mean_PD": mean_pd,
            "Mean_Diff_PD_minus_Control": mean_diff,
            "p_value": pval,
            "AUC": auc,
            "AUC_abs": auc_abs
        })

    out = pd.DataFrame(results)

    if not out.empty:
        out["p_adj_fdr"] = multipletests(out["p_value"], method="fdr_bh")[1]
        out = out.sort_values(["AUC_abs", "p_value"], ascending=[False, True]).reset_index(drop=True)

    return out


In [ ]:
erg_screen = screen_features(analysis_df, erg_features, outcome_col="PD_status")
microbiota_screen = screen_features(analysis_df, microbiota_features, outcome_col="PD_status")

print("Top ERG features")
display(erg_screen.head(10))

print("Top microbiota features")
display(microbiota_screen.head(15))


## Calculate the AUROC and confidence interval from held-out predictions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score

# ============================================================
# Matplotlib / font setup
# ============================================================
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"]  = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"]  = "Arial"
plt.style.use("default")

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

sns.set_style("white")

# ============================================================
# Inputs assumed to already exist in your workspace:
#   - erg_screen
#   - microbiota_screen
#   - analysis_df
#   - outcome_col = "PD_status"
# ============================================================
outcome_col = "PD_status"
font = "Arial"

# ============================================================
# Feature selection
# Exclude:
#   - Scoto_b_amp_mixed
#   - Scoto_OPs_amp_pure
# Then keep top 3 remaining ERG features
# ============================================================
erg_exclude = ["Scoto_b_amp_mixed", "Scoto_OPs_amp_pure"]
erg_screen_filtered = erg_screen[~erg_screen["Feature"].isin(erg_exclude)].reset_index(drop=True)

erg_features = erg_screen_filtered["Feature"].head(3).tolist()
micro_features = microbiota_screen["Feature"].head(3).tolist()
combined_features = erg_features + micro_features

print("ERG features used:", erg_features)
print("Microbiota features used:", micro_features)
print("Combined features used:", combined_features)

# ============================================================
# Complete-case dataset only
# ============================================================
all_features = list(dict.fromkeys(combined_features))
cc_df = analysis_df[all_features + [outcome_col]].dropna().copy()

X_dict = {
    "Microbiota-only": cc_df[micro_features],
}
y = cc_df[outcome_col].values

print(f"N used (complete-case): {len(cc_df)}")
print("Class counts:")
print(pd.Series(y).value_counts())

# ============================================================
# Shared settings
# ============================================================
class_counts = pd.Series(y).value_counts()
min_class_n = class_counts.min()
n_splits = min(5, min_class_n)

print(f"Using stratified {n_splits}-fold CV with random_state=42")

def make_logreg_pipeline():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(max_iter=1000, random_state=42))
    ])

# ============================================================
# Generate one pooled OOF prediction per subject
# ============================================================
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

oof_pred_dict = {}
auc_records = []

for model_name, X in X_dict.items():
    pipe = make_logreg_pipeline()
    probs = cross_val_predict(
        pipe,
        X,
        y,
        cv=cv,
        method="predict_proba"
    )[:, 1]

    oof_pred_dict[model_name] = probs
    auc_val = roc_auc_score(y, probs)

    auc_records.append({
        "Model": model_name,
        "AUROC": auc_val
    })

auc_df = pd.DataFrame(auc_records)
print("\nPooled OOF AUROC:")
display(auc_df.round(3))

# ============================================================
# Bootstrap CI for AUROC
# Paired bootstrap over subjects
# ============================================================
def bootstrap_auc_ci(y_true, y_score, n_boot=3000, seed=42):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    aucs = []
    n = len(y_true)

    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        y_b = y_true[idx]
        s_b = y_score[idx]

        if len(np.unique(y_b)) < 2:
            continue

        aucs.append(roc_auc_score(y_b, s_b))

    aucs = np.asarray(aucs)
    lo = np.percentile(aucs, 2.5)
    hi = np.percentile(aucs, 97.5)
    return lo, hi, aucs

# ============================================================
# Summarize AUROC + CI
# ============================================================
summary_rows = []
for model_name in ["Microbiota-only"]:
    auc_val = roc_auc_score(y, oof_pred_dict[model_name])
    ci_lo, ci_hi, _ = bootstrap_auc_ci(
        y,
        oof_pred_dict[model_name],
        n_boot=3000,
        seed=42
    )

    summary_rows.append({
        "Model": model_name,
        "AUROC": auc_val,
        "CI_low": ci_lo,
        "CI_high": ci_hi
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df.round(3))


## Calculate balanced accuracy and its confidence interval

In [ ]:
# ============================================================
# Balanced Accuracy using Youden threshold (OOF-based)
# ============================================================
from sklearn.metrics import balanced_accuracy_score, roc_curve

model_order = ["Microbiota-only"]

# ============================================================
# Compute Youden threshold
# ============================================================
def compute_youden_threshold(y_true, y_score):
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    j_stat = tpr - fpr
    best_idx = np.argmax(j_stat)
    return thresholds[best_idx]

# ============================================================
# Bootstrap CI
# ============================================================
def bootstrap_balacc_ci(y_true, y_score, threshold, n_boot=3000, seed=42):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    vals = []
    n = len(y_true)

    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        y_b = y_true[idx]
        s_b = y_score[idx]

        if len(np.unique(y_b)) < 2:
            continue

        preds_b = (s_b >= threshold).astype(int)
        vals.append(balanced_accuracy_score(y_b, preds_b))

    vals = np.asarray(vals)
    lo = np.percentile(vals, 2.5)
    hi = np.percentile(vals, 97.5)

    return np.mean(vals), lo, hi, vals

# ============================================================
# Bootstrap paired difference
# ============================================================
thresholds = {}
summary_rows = []

for model_name in model_order:
    probs = oof_pred_dict[model_name]

    thresh = compute_youden_threshold(y, probs)
    thresholds[model_name] = thresh

    mean_val, lo, hi, _ = bootstrap_balacc_ci(
        y,
        probs,
        threshold=thresh,
        n_boot=3000,
        seed=42
    )

    summary_rows.append({
        "Model": model_name,
        "Threshold": thresh,
        "Balanced_Accuracy": mean_val,
        "CI_low": lo,
        "CI_high": hi
    })

balacc_summary_df = pd.DataFrame(summary_rows)

print("\nBalanced Accuracy (Youden threshold):")
display(balacc_summary_df.round(3))

# ============================================================
# Pairwise comparisons


## Plot the ROC curve and confusion matrix

In [ ]:
# Visualize the same pooled out-of-fold predictions used for Figure 5b.
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(y, oof_pred_dict["Microbiota-only"])
fig, ax = plt.subplots(figsize=(5, 4.8))
ax.plot(fpr, tpr, color="#4E7EAD", linewidth=2)
ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
ax.set(xlim=(0, 1), ylim=(0, 1.02), xlabel="False positive rate",
       ylabel="True positive rate")
fig.tight_layout()
fig.savefig(notebook_output / "Fig5c_roc.pdf", bbox_inches="tight")
plt.show()


In [ ]:
# ============================================================
# Confusion Matrices (OOF predictions + Youden thresholds)
# ============================================================
from sklearn.metrics import confusion_matrix

model_order = ["Microbiota-only"]

fig, axes = plt.subplots(1, len(model_order), figsize=(5, 4.8), squeeze=False)

for i, model_name in enumerate(model_order):
    probs = oof_pred_dict[model_name]
    thresh = thresholds[model_name]

    preds = (probs >= thresh).astype(int)

    cm = confusion_matrix(y, preds)
    tn, fp, fn, tp = cm.ravel()

    ax = axes[0, i]

    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")

    # Title with threshold
    ax.set_title(
        f"{model_name}",
        fontsize=12,
        fontweight="bold",
        family=font
    )

    # Tick labels
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Ctl", "PD"], fontsize=14, fontweight="bold", family=font)
    ax.set_yticklabels(["Ctl", "PD"], fontsize=14, fontweight="bold", family=font)

    # Annotate cells (counts only, dynamic text color)
    threshold_color = cm.max() / 2.0

    for j in range(2):
        for k in range(2):
            val = cm[j, k]
            color = "white" if val > threshold_color else "black"

            ax.text(
                k, j,
                f"{val}",
                ha="center", va="center",
                fontsize=14,
                fontweight="bold",
                family=font,
                color=color
            )

    # Axis labels
    ax.set_ylabel("True label", fontsize=15, fontweight="bold", family=font)
    ax.set_xlabel("Predicted label", fontsize=15, fontweight="bold", family=font)

    # Style
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)

plt.tight_layout()
plt.savefig(notebook_output / 'Fig5d_confusion_matrix.pdf', format="pdf", bbox_inches="tight")

plt.show()


## Compare one, two and three families with and without sex

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    roc_curve
)

# ============================================================
# Settings
# ============================================================
outcome_col = "PD_status"
sex_col = "Gender"

random_state = 42
n_boot = 3000

mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"]  = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"]  = "Arial"

font = "Arial"

# ============================================================
# Feature selection
# ============================================================
erg_exclude = ["Scoto_b_amp_mixed", "Scoto_OPs_amp_pure"]

erg_screen_filtered = (
    erg_screen[
        ~erg_screen["Feature"].isin(erg_exclude)
    ]
    .reset_index(drop=True)
)

top_erg = erg_screen_filtered["Feature"].head(3).tolist()
top_micro = microbiota_screen["Feature"].head(3).tolist()

print("Top ERG features:", top_erg)
print("Top microbiota features:", top_micro)

# ============================================================
# Shared complete-case dataset
# IMPORTANT:
# Any subject missing any of the 6 selected features, sex, or outcome
# is excluded from the entire analysis.
# ============================================================
complete_case_cols = top_erg + top_micro + [sex_col, outcome_col]

cc_df = analysis_df[complete_case_cols].dropna().copy()

print(f"\nN used after complete-case filtering: {len(cc_df)}")
print("Class counts:")
print(cc_df[outcome_col].value_counts())

# ============================================================
# Pipeline helper: NO imputation
# ============================================================
def make_pipeline_no_impute(numeric_features, categorical_features):

    transformers = []

    if len(numeric_features) > 0:
        numeric_pipe = Pipeline([
            ("scaler", StandardScaler())
        ])

        transformers.append(
            ("num", numeric_pipe, numeric_features)
        )

    if len(categorical_features) > 0:
        categorical_pipe = Pipeline([
            ("onehot", OneHotEncoder(
                drop="if_binary",
                handle_unknown="ignore"
            ))
        ])

        transformers.append(
            ("cat", categorical_pipe, categorical_features)
        )

    preprocessor = ColumnTransformer(transformers)

    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("logreg", LogisticRegression(
            max_iter=1000,
            random_state=random_state
        ))
    ])

    return pipe

# ============================================================
# Bootstrap AUROC CI
# ============================================================
def bootstrap_auc_ci(y_true, y_score, n_boot=3000, seed=42):

    rng = np.random.default_rng(seed)

    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    aucs = []
    n = len(y_true)

    for _ in range(n_boot):

        idx = rng.integers(0, n, n)

        y_b = y_true[idx]
        s_b = y_score[idx]

        if len(np.unique(y_b)) < 2:
            continue

        aucs.append(
            roc_auc_score(y_b, s_b)
        )

    aucs = np.asarray(aucs)

    return (
        np.median(aucs),
        np.percentile(aucs, 2.5),
        np.percentile(aucs, 97.5)
    )

# ============================================================
# Youden threshold
# ============================================================
def compute_youden_threshold(y_true, y_score):

    fpr, tpr, thresholds = roc_curve(y_true, y_score)

    j_stat = tpr - fpr

    best_idx = np.argmax(j_stat)

    return thresholds[best_idx]

# ============================================================
# Bootstrap balanced accuracy CI
# ============================================================
def bootstrap_balacc_ci(y_true, y_score, threshold,
                        n_boot=3000, seed=42):

    rng = np.random.default_rng(seed)

    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    vals = []
    n = len(y_true)

    for _ in range(n_boot):

        idx = rng.integers(0, n, n)

        y_b = y_true[idx]
        s_b = y_score[idx]

        if len(np.unique(y_b)) < 2:
            continue

        preds_b = (s_b >= threshold).astype(int)

        vals.append(
            balanced_accuracy_score(y_b, preds_b)
        )

    vals = np.asarray(vals)

    return (
        np.median(vals),
        np.percentile(vals, 2.5),
        np.percentile(vals, 97.5)
    )

# ============================================================
# Evaluate one model
# ============================================================
def evaluate_model(df,
                   model_name,
                   feature_set_label,
                   numeric_features,
                   include_sex):

    categorical_features = [sex_col] if include_sex else []

    all_features = numeric_features + categorical_features

    X = df[all_features]

    y = df[outcome_col].astype(int).values

    class_counts = pd.Series(y).value_counts()

    n_splits = min(5, class_counts.min())

    if n_splits < 2:
        return None

    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    pipe = make_pipeline_no_impute(
        numeric_features,
        categorical_features
    )

    probs = cross_val_predict(
        pipe,
        X,
        y,
        cv=cv,
        method="predict_proba"
    )[:, 1]

    # AUROC
    auc_obs = roc_auc_score(y, probs)

    auc_median, auc_ci_low, auc_ci_high = bootstrap_auc_ci(
        y,
        probs,
        n_boot=n_boot,
        seed=random_state
    )

    # Balanced accuracy using Youden threshold
    thresh = compute_youden_threshold(y, probs)

    preds = (probs >= thresh).astype(int)

    balacc_obs = balanced_accuracy_score(y, preds)

    balacc_median, balacc_ci_low, balacc_ci_high = (
        bootstrap_balacc_ci(
            y,
            probs,
            threshold=thresh,
            n_boot=n_boot,
            seed=random_state
        )
    )

    return {
        "Model": model_name,
        "Feature_Set": feature_set_label,
        "Sex": "With sex" if include_sex else "No sex",

        "N": len(df),
        "CV_folds": n_splits,

        "AUROC": auc_obs,
        "AUROC_Median": auc_median,
        "AUROC_CI_low": auc_ci_low,
        "AUROC_CI_high": auc_ci_high,

        "Threshold": thresh,

        "Balanced_Accuracy": balacc_obs,
        "Balanced_Accuracy_Median": balacc_median,
        "Balanced_Accuracy_CI_low": balacc_ci_low,
        "Balanced_Accuracy_CI_high": balacc_ci_high
    }

# ============================================================
# Run all combinations on the SAME complete-case dataset
# ============================================================
rows = []

for include_sex in [False, True]:

    # Microbiota-only
    for m_n in [1, 2, 3]:

        rows.append(
            evaluate_model(
                cc_df,
                model_name="Microbiota-only",
                feature_set_label=f"Top {m_n} microbiota",
                numeric_features=top_micro[:m_n],
                include_sex=include_sex
            )
        )

results_df = pd.DataFrame(
    [r for r in rows if r is not None]
)

display(results_df.round(3))
